<a href="https://colab.research.google.com/github/arpitarumma/ECOPERCEPT/blob/main/finall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================
# 🌿 BERT-based Consumer Perception Analyzer
# =============================================

!pip install transformers torch pandas numpy seaborn matplotlib scikit-learn openpyxl --quiet

import pandas as pd
import numpy as np
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# 1️⃣ Load dataset
df = pd.read_excel("/content/Consumer Perceptions and Preferences for Eco-Friendly Products.xlsx")
print("✅ Dataset loaded:", df.shape)
df.head(3)

# 2️⃣ Select open-ended question columns (text columns)
text_columns = [
    "What motivates you to consider eco-friendly products?",
    "What prevents you from buying eco-friendly products more often?",
    "What kind of information would help you trust eco-friendly products more?",
    "What eco-friendly products would you like to see more of in the market?",
    "Any suggestions for brands to make eco-friendly products more attractive to you?"
]

# Combine them into one text column for overall perception
df['combined_text'] = df[text_columns].astype(str).apply(lambda x: ' '.join(x), axis=1)

# Drop empty responses
df = df[df['combined_text'].str.strip().str.len() > 10]

# 3️⃣ Auto-label sentiment using a pre-trained BERT sentiment model
from transformers import pipeline
sentiment_model = pipeline("sentiment-analysis")

print("⚙️ Generating sentiment labels...")
df['sentiment'] = df['combined_text'].apply(lambda x: sentiment_model(x[:512])[0]['label'])

# Encode sentiment labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])
label_names = le.classes_

print("\n📊 Sentiment distribution:")
print(df['sentiment'].value_counts())

# 4️⃣ Prepare data for fine-tuning
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['combined_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

# 5️⃣ Tokenize
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

class PerceptionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = PerceptionDataset(train_encodings, train_labels)
test_dataset = PerceptionDataset(test_encodings, test_labels)

# 6️⃣ Load BERT model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(label_names))

# 7️⃣ Training setup
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_dir='./logs',
    logging_steps=100,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# 8️⃣ Train model
trainer.train()

# 9️⃣ Evaluate
preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)

acc = accuracy_score(test_labels, y_pred)
print(f"\n🎯 Final Test Accuracy: {acc*100:.2f}%")
print("\n📋 Classification Report:")
print(classification_report(test_labels, y_pred, target_names=label_names))

# 🔟 Confusion matrix
cm = confusion_matrix(test_labels, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("BERT Sentiment Confusion Matrix")
plt.show()

print("\n🏁 Done! Consumer perception analyzed successfully.")


✅ Dataset loaded: (1008, 21)


KeyError: "['What motivates you to consider eco-friendly products?', 'What prevents you from buying eco-friendly products more often?', 'What eco-friendly products would you like to see more of in the market?', 'Any suggestions for brands to make eco-friendly products more attractive to you?'] not in index"

In [ ]:
# =============================================
# Consumer Purchase Prediction: BERT + Features
# =============================================

!pip install -q transformers sentence-transformers catboost scikit-learn pandas numpy openpyxl

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer
import joblib

# ============================
# Step 1. Load Dataset
# ============================
data = pd.read_excel("/content/Consumer Perceptions and Preferences for Eco-Friendly Products.xlsx")

# ============================
# Step 2. Target variable
# ============================
target_col = "Have you ever purchased an eco-friendly/sustainable product?"
data['target'] = data[target_col].astype(str).str.strip().str.lower().map(
    lambda x: 1 if 'yes' in x else (0 if 'no' in x else np.nan)
)
data = data.dropna(subset=['target']).reset_index(drop=True)
data['target'] = data['target'].astype(int)

# Remove unneeded columns
for c in ['Timestamp','Name']:
    if c in data.columns:
        data = data.drop(columns=c)

# ============================
# Step 3. Numeric mappings
# ============================
def try_map(src_col, mapping):
    if src_col not in data.columns:
        return pd.Series([np.nan]*len(data))
    s = data[src_col].astype(str).map(lambda x: mapping.get(x.strip(), np.nan) if isinstance(x,str) else np.nan)
    mask = s.isna()
    if mask.any():
        digits = data.loc[mask, src_col].astype(str).str.extract(r'(\d+)').astype(float)
        s.loc[mask] = digits[0].values
    return s

importance_map = {'Very important':5, 'Important':4, 'Somewhat important':3,
                  'Not so important':2, 'Not at all important':1, 'Extremely important':5, 'Neutral':3}
recommend_map = {'Extremely likely':5, 'Very likely':4, 'Somewhat likely':3,
                 'Not so likely':2, 'Not at all likely':1, 'Likely':4, 'Neutral':3}
frequency_map = {'Daily':7, 'Weekly':6, 'Monthly':5, 'Quarterly':4, 'A few times a year':3,
                 'Once a year':2, 'Never':0, 'Often':5, 'Sometimes':3, 'Rarely':1}

data['importance_num'] = try_map("How important is sustainability in your purchasing decisions?", importance_map)
data['recommend_num'] = try_map("How likely are you to recommend eco-friendly products to your friends/family?", recommend_map)
data['freq_num'] = try_map("How often do you buy eco-friendly products?", frequency_map)

# ============================
# Step 4. Multi-hot categories
# ============================
ms_col = "If yes, what categories have you purchased from? (Select all that apply)"
multi_hot_df = pd.DataFrame(index=data.index)
if ms_col in data.columns:
    split_series = data[ms_col].fillna('').astype(str).apply(
        lambda x: [s.strip() for s in re.split(r',|;|\||/', x) if s.strip()]
    )
    cats = sorted(set([c for lst in split_series for c in lst if c]))
    for cat in cats:
        col_name = "cat__" + re.sub(r'\W+', '_', cat)[:60]
        multi_hot_df[col_name] = split_series.apply(lambda lst: 1 if cat in lst else 0)
    data = pd.concat([data, multi_hot_df], axis=1)

# ============================
# Step 5. Text preprocessing
# ============================
text_cols = [c for c in [
    "What motivates you to consider eco-friendly products? ",
    "What kind of information would help you trust eco-friendly products more?",
    "What eco-friendly products would you like to see more of in the market? ",
    "Any suggestions for brands to make eco-friendly products more attractive to you? ",
    "What prevents you from buying eco-friendly products more often? "
] if c in data.columns]

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['combined_text'] = data[text_cols].fillna('').apply(lambda x: ' '.join([clean_text(t) for t in x]), axis=1)

# ============================
# Step 6. Generate sentence embeddings (BERT)
# ============================
print("⚙️ Generating sentence embeddings...")
model_emb = SentenceTransformer('all-MiniLM-L6-v2')
text_embeddings = model_emb.encode(data['combined_text'].tolist(), batch_size=32, show_progress_bar=True)

# ============================
# Step 7. Combine with numeric + multi-hot features
# ============================
numeric_cols = ['importance_num','recommend_num','freq_num']
mh_cols = [c for c in data.columns if c.startswith('cat__')]
all_features = np.hstack([
    text_embeddings,
    data[numeric_cols + mh_cols].fillna(0).values
])

# Scale numeric + multi-hot
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
all_features[:, -len(numeric_cols + mh_cols):] = scaler.fit_transform(all_features[:, -len(numeric_cols + mh_cols):])

# ============================
# Step 8. Train/Test Split
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    all_features, data['target'].values, test_size=0.2, stratify=data['target'], random_state=42
)

# ============================
# Step 9. Train CatBoost on combined features
# ============================
clf = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='Accuracy',
    verbose=50,
    random_seed=42,
    class_weights=[1.0, sum(y_train==0)/sum(y_train==1)]  # balance minority
)

clf.fit(X_train, y_train, eval_set=(X_test, y_test))

# ============================
# Step 10. Evaluation
# ============================
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ Hybrid BERT + Features CatBoost Accuracy: {acc:.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ============================
# Step 11. Save Model
# ============================
joblib.dump({'model': clf, 'scaler': scaler, 'embed_model': model_emb}, "/content/eco_model_bert_catboost.pkl")
print("\nModel saved as eco_model_bert_catboost.pkl")


⚙️ Generating sentence embeddings...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

0:	learn: 0.6125974	test: 0.5102116	best: 0.5102116 (0)	total: 126ms	remaining: 2m 6s
50:	learn: 0.9296819	test: 0.6079940	best: 0.6079940 (50)	total: 6s	remaining: 1m 51s
100:	learn: 0.9915454	test: 0.6693503	best: 0.6693503 (91)	total: 10.6s	remaining: 1m 34s
150:	learn: 1.0000000	test: 0.6923789	best: 0.6923789 (149)	total: 15.2s	remaining: 1m 25s
200:	learn: 1.0000000	test: 0.6955029	best: 0.7093201 (174)	total: 21s	remaining: 1m 23s
250:	learn: 1.0000000	test: 0.6947621	best: 0.7093201 (174)	total: 25.7s	remaining: 1m 16s
300:	learn: 1.0000000	test: 0.7032327	best: 0.7177907 (289)	total: 31.5s	remaining: 1m 13s
350:	learn: 1.0000000	test: 0.7085793	best: 0.7177907 (289)	total: 36.2s	remaining: 1m 6s
400:	learn: 1.0000000	test: 0.7131850	best: 0.7231373 (390)	total: 40.9s	remaining: 1m 1s
450:	learn: 1.0000000	test: 0.7124441	best: 0.7284839 (440)	total: 46.6s	remaining: 56.8s
500:	learn: 1.0000000	test: 0.7231373	best: 0.7284839 (440)	total: 51.3s	remaining: 51.1s
550:	learn: 1.00

In [ ]:
# =============================================
# Consumer Purchase Prediction - Hybrid Features + CatBoost
# =============================================

!pip install -q catboost sentence-transformers transformers scikit-learn pandas numpy openpyxl

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import joblib

# ============================
# Step 1. Load Dataset
# ============================
data = pd.read_excel("/content/Consumer Perceptions and Preferences for Eco-Friendly Products.xlsx")

# ============================
# Step 2. Target variable
# ============================
target_col = "Have you ever purchased an eco-friendly/sustainable product?"
data['target'] = data[target_col].astype(str).str.strip().str.lower().map(
    lambda x: 1 if 'yes' in x else (0 if 'no' in x else np.nan)
)
data = data.dropna(subset=['target']).reset_index(drop=True)
data['target'] = data['target'].astype(int)

# Remove unneeded columns
for c in ['Timestamp','Name']:
    if c in data.columns:
        data = data.drop(columns=c)

# ============================
# Step 3. Numeric mappings
# ============================
def try_map(src_col, mapping):
    if src_col not in data.columns:
        return pd.Series([np.nan]*len(data))
    s = data[src_col].astype(str).map(lambda x: mapping.get(x.strip(), np.nan) if isinstance(x,str) else np.nan)
    mask = s.isna()
    if mask.any():
        digits = data.loc[mask, src_col].astype(str).str.extract(r'(\d+)').astype(float)
        s.loc[mask] = digits[0].values
    return s

importance_map = {'Very important':5, 'Important':4, 'Somewhat important':3,
                  'Not so important':2, 'Not at all important':1, 'Extremely important':5, 'Neutral':3}
recommend_map = {'Extremely likely':5, 'Very likely':4, 'Somewhat likely':3,
                 'Not so likely':2, 'Not at all likely':1, 'Likely':4, 'Neutral':3}
frequency_map = {'Daily':7, 'Weekly':6, 'Monthly':5, 'Quarterly':4, 'A few times a year':3,
                 'Once a year':2, 'Never':0, 'Often':5, 'Sometimes':3, 'Rarely':1}

data['importance_num'] = try_map("How important is sustainability in your purchasing decisions?", importance_map)
data['recommend_num'] = try_map("How likely are you to recommend eco-friendly products to your friends/family?", recommend_map)
data['freq_num'] = try_map("How often do you buy eco-friendly products?", frequency_map)

# ============================
# Step 4. Multi-hot categories
# ============================
ms_col = "If yes, what categories have you purchased from? (Select all that apply)"
multi_hot_df = pd.DataFrame(index=data.index)
if ms_col in data.columns:
    split_series = data[ms_col].fillna('').astype(str).apply(
        lambda x: [s.strip() for s in re.split(r',|;|\||/', x) if s.strip()]
    )
    cats = sorted(set([c for lst in split_series for c in lst if c]))
    for cat in cats:
        col_name = "cat__" + re.sub(r'\W+', '_', cat)[:60]
        multi_hot_df[col_name] = split_series.apply(lambda lst: 1 if cat in lst else 0)
    data = pd.concat([data, multi_hot_df], axis=1)

mh_cols = [c for c in data.columns if c.startswith('cat__')]

# ============================
# Step 5. Text preprocessing
# ============================
text_cols = [c for c in [
    "What motivates you to consider eco-friendly products? ",
    "What kind of information would help you trust eco-friendly products more?",
    "What eco-friendly products would you like to see more of in the market? ",
    "Any suggestions for brands to make eco-friendly products more attractive to you? ",
    "What prevents you from buying eco-friendly products more often? "
] if c in data.columns]

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['combined_text'] = data[text_cols].fillna('').apply(lambda x: ' '.join([clean_text(t) for t in x]), axis=1)

# ============================
# Step 6. Sentence embeddings
# ============================
print("⚙️ Generating sentence embeddings...")
model_emb = SentenceTransformer('all-MiniLM-L6-v2')
text_embeddings = model_emb.encode(data['combined_text'].tolist(), batch_size=32, show_progress_bar=True)

# ============================
# Step 7. Engineered features
# ============================
from transformers import pipeline
sentiment_model = pipeline("sentiment-analysis", device=0)  # use GPU if available

data['eco_score'] = data[['importance_num','recommend_num','freq_num']].mean(axis=1)
data['text_length'] = data['combined_text'].apply(lambda x: len(str(x).split()))
data['eco_keywords'] = data['combined_text'].apply(lambda x: sum(x.lower().count(k) for k in ['eco','sustain','green','organic']))
data['num_categories'] = data[mh_cols].sum(axis=1)
# Sentiment scoring (can take time)
data['sentiment_score'] = data['combined_text'].apply(lambda x: sentiment_model(x[:512])[0]['score'])

engineered_cols = ['eco_score','text_length','eco_keywords','sentiment_score','num_categories']

# ============================
# Step 8. Combine features
# ============================
import numpy as np
all_features = np.hstack([text_embeddings, data[engineered_cols + mh_cols].fillna(0).values])

# Scale numeric + multi-hot
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
all_features[:, -len(engineered_cols + mh_cols):] = scaler.fit_transform(all_features[:, -len(engineered_cols + mh_cols):])

# ============================
# Step 9. Train/Test Split
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    all_features, data['target'].values, test_size=0.2, stratify=data['target'], random_state=42
)

# ============================
# Step 10. Train CatBoost
# ============================
clf = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='Accuracy',
    verbose=50,
    random_seed=42,
    class_weights=[1.0, sum(y_train==0)/sum(y_train==1)]
)

clf.fit(X_train, y_train, eval_set=(X_test, y_test))

# ============================
# Step 11. Evaluation
# ============================
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ Final Hybrid Model Accuracy: {acc:.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ============================
# Step 12. Save Model
# ============================
joblib.dump({'model': clf, 'scaler': scaler, 'embed_model': model_emb}, "/content/eco_model_full_hybrid.pkl")
print("\nModel saved as eco_model_full_hybrid.pkl")


⚙️ Generating sentence embeddings...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


0:	learn: 0.7783777	test: 0.7994715	best: 0.7994715 (0)	total: 128ms	remaining: 2m 7s
50:	learn: 0.9639927	test: 0.7163090	best: 0.7994715 (0)	total: 4.78s	remaining: 1m 28s
100:	learn: 1.0000000	test: 0.6964043	best: 0.7994715 (0)	total: 10.7s	remaining: 1m 35s
150:	learn: 1.0000000	test: 0.7017509	best: 0.7994715 (0)	total: 15.3s	remaining: 1m 26s
200:	learn: 1.0000000	test: 0.6964043	best: 0.7994715 (0)	total: 20.1s	remaining: 1m 19s
250:	learn: 1.0000000	test: 0.6910577	best: 0.7994715 (0)	total: 25.9s	remaining: 1m 17s
300:	learn: 1.0000000	test: 0.6857111	best: 0.7994715 (0)	total: 30.6s	remaining: 1m 11s
350:	learn: 1.0000000	test: 0.6910577	best: 0.7994715 (0)	total: 37.7s	remaining: 1m 9s
400:	learn: 1.0000000	test: 0.6903168	best: 0.7994715 (0)	total: 42.5s	remaining: 1m 3s
450:	learn: 1.0000000	test: 0.6956635	best: 0.7994715 (0)	total: 47.2s	remaining: 57.5s
500:	learn: 1.0000000	test: 0.7002692	best: 0.7994715 (0)	total: 53.2s	remaining: 53s
550:	learn: 1.0000000	test: 0.7